# Analyze Sweep Results

This notebook inspects the sweep RMSE results exported during training.


In [ ]:
from pathlib import Path
import pandas as pd

SWEEP_CSV = Path("../outputs/to_export_sweep/sweep_results.csv")
assert SWEEP_CSV.exists(), f"Missing sweep file: {SWEEP_CSV}"

df = pd.read_csv(SWEEP_CSV)
df.head()


In [ ]:
head1_labels = [
    "rocof_max_COI",
    "rocof_min_COI",
    "devup_COI",
    "devdown_COI",
]
head2_labels = [
    "Delta_P_IBR_1",
    "Delta_P_IBR_2",
    "Delta_P_IBR_3",
    "Delta_P_IBR_4",
]


def _avg_for(labels, label_name):
    subset = df[df["label"].isin(labels)]
    return (
        subset.groupby(["model", "loss", "scaler"], as_index=False)[["rmse", "norm"]]
        .mean()
        .assign(label=label_name)
    )


avg_all = (
    df.groupby(["model", "loss", "scaler"], as_index=False)[["rmse", "norm"]]
    .mean()
    .assign(label="ALL")
)
avg_head1 = _avg_for(head1_labels, "HEAD1")
avg_head2 = _avg_for(head2_labels, "HEAD2")

summary = pd.concat([avg_all, avg_head1, avg_head2], ignore_index=True)
df_with_avg = pd.concat([df, summary], ignore_index=True)

summary.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plot_summary = (
    summary.groupby(["model", "loss", "label"], as_index=False)[["rmse", "norm"]]
    .mean()
    .sort_values(["model", "loss", "label"])
)

g = sns.catplot(
    data=plot_summary,
    x="label",
    y="rmse",
    hue="loss",
    col="model",
    kind="bar",
    col_wrap=3,
    height=3,
    aspect=1.1,
)
for ax in g.axes.flatten():
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()


In [ ]:
table_df = summary.sort_values(["model", "loss", "scaler", "label"])
latex = table_df.to_latex(index=False, float_format="%.4f")
print(latex)


In [ ]:
df


In [ ]:
table_df = (
    df.groupby(["model", "scaler", "loss", "label"], as_index=False)[["rmse"]]
    .min()
    .sort_values("rmse")
)

table_df
